# DLS walkthrough: 3HAE annealed samples

This executable example shows how the analysis estimates polydispersity and hydrodynamic size from a Dynamics `.exp` export. It explains both the calculations and their limits: a mathematically successful fit is not automatically a trustworthy particle-size measurement.

The selected file contains two instrument measurement blocks with 40 acquisitions each. The notebook keeps acquisitions and samples distinct: it uses the per-sample median curve only for visualization and fits each acquisition separately. Parsing, fitting, size-estimation, plotting, and summary functions are implemented in the adjacent `dls_walkthrough.py` module; the cells below make those operations inspectable and explain why they are performed.

**Workflow:** confirm the input and metadata → inspect raw correlation curves and their variation over time → fit and audit each acquisition → compare accepted PDI and hydrodynamic-radius estimates → review assumptions and the equations in the mathematical appendix. Run the notebook from top to bottom so later cells use the tables and results created earlier.

## 1. Set up the analysis and choose the input

The setup cell locates the DLS folder relative to either the notebook's current working directory or the workspace root, adds that folder to Python's import path, and imports the analysis helpers. The selected `.exp` path is explicit: changing `DATA_PATH` is the main step needed to reuse this example for another instrument export. The display and plotting imports are used only for notebook presentation; fitting and parsing are delegated to `dls_walkthrough.py`.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.rcParams["figure.dpi"] = 130

DATA_FILENAME = "20261103_3HAE-annealed.exp"
WORKING_DIR = Path.cwd().resolve()
DLS_CANDIDATES = [
    WORKING_DIR,
    *WORKING_DIR.parents,
    WORKING_DIR / "origami_data" / "DLS",
    WORKING_DIR / "DLS",
]
DLS_DIR = next(
    (candidate for candidate in DLS_CANDIDATES if (candidate / "data" / DATA_FILENAME).is_file()),
    None,
)
if DLS_DIR is None:
    raise FileNotFoundError(
        f"Could not locate data/{DATA_FILENAME} from the current directory: {WORKING_DIR}"
    )
DATA_PATH = DLS_DIR / "data" / DATA_FILENAME
PLOT_DIR = DLS_DIR / "plots" / DATA_PATH.stem
PLOT_DIR.mkdir(parents=True, exist_ok=True)
if str(DLS_DIR) not in sys.path:
    sys.path.insert(0, str(DLS_DIR))

from dls_walkthrough import (
    analyze_hydrodynamic_radii,
    load_exp_file,
    measurement_overview,
    plot_accepted_polydispersity,
    plot_autocorrelations_by_time,
    plot_median_correlation_comparison,
    plot_autocorrelations,
    plot_hydrodynamic_radii,
    summarize_exclusions,
    summarize_hydrodynamic_radii,
)

print(f"Data file: {DATA_PATH}")
print(f"Plots will be saved to: {PLOT_DIR}")

## 2. Parse the instrument export and verify what will be analyzed

`load_exp_file` reads the laser wavelength, lag-time vector, measurement names, acquisition IDs, elapsed measurement times, recorded temperatures, and each `c1` correlation curve. `measurement_overview` then summarizes the blocks before any fitting. This is a deliberate quality-control step: confirm that the expected two blocks and 40 acquisitions per block are present, that temperatures and time ranges are plausible, and that the curve lengths agree. In this export the parser finds 80 curves, 399 lag points per curve, a wavelength of 662.83 nm, and temperatures near 25 °C.

Use the measurement names as recorded by the instrument. The second block's label (`RNA-origami_25C_2.99e+003s_2`) does not establish a sample identity by itself, so this notebook does not relabel it as 5HTB without external experimental metadata. The 30-second acquisition duration shown by the overview is descriptive metadata; each curve is still processed independently.

In [2]:
experiment = load_exp_file(DATA_PATH)
overview = pd.DataFrame(measurement_overview(experiment))

display(overview)
print(f"Laser wavelength: {experiment.wavelength_nm:.2f} nm")
print(
    "Recorded acquisition temperatures (°C):",
    sorted({round(item.temperature_c, 2) for item in experiment.acquisitions}),
)

,measurement_id,sample,measurement,acquisitions,acquisition_duration_s,temperature_min_c,temperature_max_c,elapsed_start_s,elapsed_end_s,curve_points
0,1,Sample 1,RNA-origami_25C_3HAE_annealed,40,30,24.9887,24.9999,29.4,1247.7,399
1,2,Sample 2,RNA-origami_25C_2.99e+003s_2,40,30,24.9995,25.0014,1776.4,2994.9,399


Laser wavelength: 662.83 nm
Recorded acquisition temperatures (°C): [24.99, 25.0]


## 3. Inspect the exported autocorrelation curves

Each faint line is one acquisition's exported `c1` curve; the darker line is the pointwise median for that measurement block. The logarithmic lag-time axis makes both the early decay and the long-lag tail visible. Inspecting the raw curves first helps reveal baseline offsets, noisy tails, unusually slow or fast decays, and acquisition-to-acquisition variability before reducing them to a fitted number.

These panels are diagnostic only. The median line is not used as a fit input, and curves are not pooled across samples. Baseline estimation, normalization, smoothing, and fit-window selection are applied separately to each acquisition later.

In [3]:
correlation_figure, correlation_axes = plot_autocorrelations(experiment)

for ax, row in zip(correlation_axes.flat, overview.itertuples()):
    ax.set_title(row.measurement)

correlation_figure.savefig(
    PLOT_DIR / f"{DATA_PATH.stem}_autocorrelations.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

### Compare the typical curve shapes

This overlay shows the pointwise median exported curve for each measurement block on common axes, making differences in typical decay shape easier to see. It is a visual comparison, not a pooled curve or a substitute for acquisition-level fitting: a median curve can hide multimodality and does not preserve the distribution of individual fit outcomes.

In [4]:
median_correlation_figure, median_correlation_axis = (
    plot_median_correlation_comparison(experiment)
)
for line, row in zip(median_correlation_axis.lines, overview.itertuples()):
    line.set_label(row.measurement)
median_correlation_axis.legend(frameon=False)
median_correlation_figure.savefig(
    PLOT_DIR / f"{DATA_PATH.stem}_median_autocorrelation.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

### Check whether curves change over measurement time

Each acquisition curve is colored by its elapsed measurement time; the black curve is the block's pointwise median. A systematic color progression in decay shape, baseline, or scatter can suggest drift, aggregation, settling, or changing measurement conditions and can guide follow-up inspection. It does not identify a cause or automatically reject an acquisition, so interpret it alongside the fit audit and experimental notes.

In [5]:
time_correlation_figure, time_correlation_axes = (
    plot_autocorrelations_by_time(experiment)
)
for axis, row in zip(time_correlation_axes, overview.itertuples()):
    axis.set_title(row.measurement)
time_correlation_figure.savefig(
    PLOT_DIR / f"{DATA_PATH.stem}_autocorrelations_by_time.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

## 4. Fit every acquisition and make explicit acceptance decisions

The helper processes all 80 acquisitions independently. For each curve it estimates a long-lag baseline and initial amplitude, converts the baseline-corrected intensity correlation to a normalized field-correlation estimate using the square-root Siegert approximation, smooths that estimate with a five-point moving average, and selects the portion between its first crossings below 0.8 and then below 0.2. It regresses the natural logarithm of the selected values against a quadratic function of lag time. The linear and quadratic coefficients give the first and second cumulants; from these the code estimates a decay rate and PDI. When a positive decay rate is available, the code converts it to a diffusion coefficient and then a hydrodynamic radius using the recorded temperature and the optical and solvent assumptions documented in the appendix.

The checks prevent undefined or uninformative calculations: paired correlation/lag arrays must contain at least 30 points; the estimated amplitude must be positive; the curve must span the fit-window thresholds; the window must have at least eight points; lag times must be positive; and the regression must have nonzero variation. The resulting decay rate must be positive and PDI finite. A calculable fit is then accepted for summaries only when `0 <= PDI <= 1` and `R² >= 0.80`. These are transparent screening rules, not universal physical laws or proof that the sample is monodisperse. The `R²` cutoff is an analysis choice and should be checked against the data and instrument practice.

Rejected acquisitions remain in the audit. If a fit produced a decay rate but failed a final PDI or `R²` gate, its diagnostic radius is retained and shown as rejected; it is not included in accepted-fit medians. If the curve cannot produce a usable decay rate, its radius is missing. The exclusion summary assigns one first-applicable reason per rejected acquisition. Confirm the instrument's `c1` convention and lag-time units before interpreting these model-based estimates; a good-looking regression cannot correct a wrong export convention or unit.

In [6]:
MIN_R_SQUARED = 0.80
SCATTERING_ANGLE_DEG = 90.0
REFRACTIVE_INDEX = 1.333
VISCOSITY_PA_S = None

fit_results = analyze_hydrodynamic_radii(
    experiment,
    min_r_squared=MIN_R_SQUARED,
    scattering_angle_deg=SCATTERING_ANGLE_DEG,
    refractive_index=REFRACTIVE_INDEX,
    viscosity_pa_s=VISCOSITY_PA_S,
)
fit_table = pd.DataFrame(fit_results)
fit_summary = pd.DataFrame(summarize_hydrodynamic_radii(fit_results))
exclusion_table = pd.DataFrame(summarize_exclusions(fit_results))

print("Per-sample accepted-size summary")
display(fit_summary)

print("Rejected acquisitions by reason")
display(exclusion_table)

Per-sample accepted-size summary


,measurement_id,sample,measurement,temperature_mean_c,acquisitions,fits_accepted,fits_rejected,median_accepted_Rh_nm,median_accepted_diameter_nm
0,1,Sample 1,RNA-origami_25C_3HAE_annealed,24.996787,40,6,34,190211.317094,380422.634188
1,2,Sample 2,RNA-origami_25C_2.99e+003s_2,25.000095,40,30,10,114.757578,229.515155


Rejected acquisitions by reason


,sample,measurement,exclusion_reason,acquisitions_excluded,radius_calculated_but_excluded
0,Sample 1,RNA-origami_25C_3HAE_annealed,PDI outside 0-1 fit range,11,11
1,Sample 1,RNA-origami_25C_3HAE_annealed,fit R-squared below threshold,23,23
2,Sample 2,RNA-origami_25C_2.99e+003s_2,fit R-squared below threshold,10,10


## 5. Audit each acquisition and the exclusion counts

The sample-level summary reports how many fits passed the gates and summarizes accepted radii; the exclusion table counts rejected acquisitions by reason and shows how many nevertheless had a calculable radius. The acquisition-level table below keeps the decision traceable to measurement and acquisition IDs, with elapsed time, temperature, PDI, fit `R²`, radius, acceptance flag, and rejection reason. A missing radius means a valid positive decay rate could not be calculated; a rejected row with a radius failed a later quality gate.

In [7]:
display(
    fit_table.set_index(["sample", "acquisition"])[
        [
            "elapsed_s",
            "temperature_c",
            "pdi",
            "fit_r_squared",
            "hydrodynamic_radius_nm",
            "hydrodynamic_diameter_nm",
            "fit_accepted",
            "rejection_reason",
        ]
    ].round(3)
)

elapsed_s  temperature_c    pdi  fit_r_squared  \
sample   acquisition                                                   
Sample 1 1                 29.4         24.989  0.432          0.928   
         2                 59.4         24.990  0.477          0.667   
         3                 89.4         24.990  0.750          0.619   
         4                119.4         24.991  0.499          0.473   
         5                149.4         24.992  0.426          0.955   
...                         ...            ...    ...            ...   
Sample 2 36              2842.5         25.000  0.476          0.696   
         37              2904.8         25.000  0.419          0.980   
         38              2934.8         25.000  0.443          0.967   
         39              2964.9         25.000  0.477          0.863   
         40              2994.9         25.001  0.396          0.990   

                      hydrodynamic_radius_nm  hydrodynamic_diameter_nm  \
sample   acquisition                                                     
Sample 1 1                            43.812                    87.624   
         2                        245626.107                491252.214   
         3                        342670.490                685340.979   
         4                         47455.468                 94910.937   
         5                          3177.481                  6354.962   
...                                      ...                       ...   
Sample 2 36                        23548.606                 47097.211   
         37                          111.223                   222.445   
         38                         3315.853                  6631.705   
         39                         2641.520                  5283.040   
         40                           67.147                   134.294   

                      fit_accepted               rejection_reason  
sample   acquisition                                               
Sample 1 1                    True                           None  
         2                   False  fit R-squared below threshold  
         3                   False  fit R-squared below threshold  
         4                   False  fit R-squared below threshold  
         5                    True                           None  
...                            ...                            ...  
Sample 2 36                  False  fit R-squared below threshold  
         37                   True                           None  
         38                   True                           None  
         39                   True                           None  
         40                   True                           None  

[80 rows x 8 columns]

## 6. Compare acquisition-level hydrodynamic radii

The logarithmic radius axis is used because individual estimates can span orders of magnitude. Circles are accepted acquisitions; crosses are rejected acquisitions for which a radius could still be calculated. A rejected point is shown for diagnosis, not counted as a valid sample-size result. The dotted line and annotation give the median radius among accepted fits in that measurement block only. Compare the number of accepted points with the total acquisitions before interpreting the median: a median based on very few accepted curves is not a robust sample characterization.

In [8]:
radius_figure, radius_axes = plot_hydrodynamic_radii(fit_results)
radius_figure.savefig(
    PLOT_DIR / f"{DATA_PATH.stem}_hydrodynamic_radii.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

## 7. Compare accepted-fit PDI over elapsed measurement time

Each panel shows PDI estimates for accepted acquisitions from one measurement block, in elapsed-time order. The dotted line is the median PDI of those accepted fits. Looking at individual values as well as the median shows whether a summary conceals time trends or scatter. Rejected fits are intentionally omitted from this plot; consult the audit and raw-curve views to see why they were excluded.

In [9]:
pdi_figure, pdi_axes = plot_accepted_polydispersity(fit_results)
pdi_figure.savefig(
    PLOT_DIR / f"{DATA_PATH.stem}_accepted_pdi.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

## 8. Interpret the results with their limitations

Both measurement blocks in the selected file were recorded near 25 °C, with 40 acquisitions per block, so this example compares blocks at one temperature; it is not a temperature comparison. The calculated values are model-based estimates from exported correlations, not instrument-reported sizes. Their validity depends on the correlation convention and lag-time units, the fit-window heuristic, scattering angle, refractive index, and solvent viscosity. Confirm these against the instrument configuration and experimental record before making quantitative claims.

Assess fit counts and acquisition-level scatter alongside the median. In particular, if a block has few accepted fits or widely varying radii, report that uncertainty and inspect its raw curves rather than treating its median as a definitive particle size. DLS weights scattering intensity strongly, so a small number of large scatterers can influence the signal; PDI and a cumulants fit do not identify particle composition or establish a number-weighted size distribution. Independent sample replicates and appropriate uncertainty analysis are needed for broader conclusions.

## Mathematical appendix 1: what the correlation curve represents

In dynamic light scattering, Brownian motion changes the phase of scattered light. The detector's temporal intensity fluctuations are summarized by an intensity autocorrelation function. For a common normalized convention, the Siegert relation is

$$g_2(\tau) = 1 + \beta\,|g_1(\tau)|^2,$$

where $g_2$ is the normalized intensity correlation, $g_1$ is the normalized electric-field correlation, $\tau$ is lag time, and $\beta$ is an optical coherence factor. The instrument export's `c1` label is format-specific; the calculation below assumes its baseline and amplitude can be treated as an intensity-correlation curve with a decaying excess signal. Confirm that assumption for the instrument/export in use.

For one acquisition, the code estimates a tail baseline $b$ using the median of the final 30 exported points and an excess amplitude $A$ using the median of the first 10 points minus that baseline:

$$b=\operatorname{median}(c_1[-30:]), \qquad A=\operatorname{median}(c_1[:10])-b.$$

It then forms the normalized decay estimate

$$\widehat{g}_1(\tau_i)=\sqrt{\max\!\left(\frac{c_1(\tau_i)-b}{A},10^{-8}\right)}.$$

This is the square-root Siegert approximation after baseline subtraction and normalization; normalization removes the unknown amplitude/coherence scale. A non-positive $A$ means there is no usable positive decay amplitude and the acquisition is rejected. The lower clipping bound prevents taking the logarithm of zero or a negative number later, but it does not restore lost signal: a curve whose tail is clipped or noisy still needs critical inspection.

The code applies a five-point moving average to reduce point-to-point noise before selecting a fit interval. It uses the first smoothed value below 0.8 as the start and the first subsequent value below 0.2 as the end (the end point itself is excluded). These thresholds choose a reproducible portion of the decay, not a universal DLS standard. Because the moving window is defined in array indices, it does not represent a fixed lag-time width when lag points are nonuniform.

## Mathematical appendix 2: cumulants, PDI, and fit quality

For the selected lag times $\tau_i$, the implementation scales lag time by its median $s=\operatorname{median}(\tau_i)$, so $x_i=\tau_i/s$ is dimensionless, and sets $y_i=\ln\widehat{g}_1(\tau_i)$. A least-squares quadratic is fitted:

$$y_i=a_0+a_1x_i+a_2x_i^2.$$

The cumulants expansion for a distribution of decay rates is, to second order,

$$\ln g_1(\tau)\simeq -\kappa_1\tau+\frac{\kappa_2}{2}\tau^2,$$

where $\kappa_1$ is the mean decay rate (s$^{-1}$) and $\kappa_2$ describes the variance of decay rates (s$^{-2}$). Matching coefficients while accounting for the scale $s$ gives

$$\Gamma=\kappa_1=-\frac{a_1}{s}, \qquad \kappa_2=\frac{2a_2}{s^2}, \qquad \mathrm{PDI}=\frac{\kappa_2}{\kappa_1^2}=\frac{2a_2}{a_1^2}.$$

Thus $\Gamma$ has units s$^{-1}$, while PDI is dimensionless. In this cumulants model, PDI is a normalized breadth of the decay-rate distribution; it is not a direct measurement of a particle-size histogram. A negative or out-of-range estimate can arise from noise, baseline errors, an unsuitable window, or model mismatch and is not silently forced into the physical range for fitting or reporting.

Fit quality is calculated on the selected log-correlation points as

$$R^2=1-\frac{\sum_i(y_i-\widehat y_i)^2}{\sum_i(y_i-\overline y)^2}.$$

A usable regression also requires nonzero variation in $y$, at least eight fit points, and positive lag times. The analysis retains a finite estimate for diagnosis but accepts it for sample summaries only if $\Gamma>0$, $0\leq\mathrm{PDI}\leq1$, and $R^2\geq0.80$. The threshold is configurable in the code and is a screening convention, not a confidence interval or a guarantee of a physically correct model. Each rejected acquisition is counted once using the first applicable reason.

## Mathematical appendix 3: from decay rate to hydrodynamic size

A translationally diffusing particle causes the field correlation to decay approximately as $g_1(\tau)=\exp(-\Gamma\tau)$ for a single decay rate. At scattering-vector magnitude $q$, the rate and diffusion coefficient are related by

$$\Gamma=Dq^2, \qquad D=\frac{\Gamma}{q^2}.$$

For vacuum wavelength $\lambda$, medium refractive index $n$, and scattering angle $\theta$, the scattering-vector magnitude is

$$q=\frac{4\pi n}{\lambda}\sin\!\left(\frac{\theta}{2}\right).$$

Use consistent SI units: the code converts the wavelength from nanometres to metres, so $q$ is in m$^{-1}$, $\Gamma$ in s$^{-1}$, and $D$ in m$^2$/s. The Stokes–Einstein relation for a spherical equivalent particle undergoing translational diffusion in a Newtonian fluid is

$$D=\frac{k_BT}{6\pi\eta R_h}, \qquad R_h=\frac{k_BT}{6\pi\eta D}, \qquad d_h=2R_h=\frac{k_BT}{3\pi\eta D}.$$

Here $R_h$ is hydrodynamic radius, $d_h$ is diameter, $k_B=1.380649\times10^{-23}$ J/K, $T=T_{\!^\circ\mathrm C}+273.15$ is absolute temperature, and $\eta$ is dynamic viscosity in Pa·s. The result is a hydrodynamic equivalent size (including solvent coupling), not necessarily the geometric diameter of a rigid object. Applying Stokes–Einstein assumes the measured decay represents translational Brownian diffusion and that the viscosity and optical geometry are appropriate.

This notebook uses the wavelength recorded in the `.exp` file, the acquisition's recorded temperature, 90° scattering, refractive index 1.333, and a water-viscosity approximation interpolated between 25 °C and 37 °C. These are assumptions to verify or replace for another setup. The helper's linear interpolation clamps to the endpoint values outside that temperature range; it should not be treated as a general solvent-viscosity model.

## Mathematical appendix 4: aggregation, plots, and exported results

For measurement block $j$, let $\mathcal A_j$ be the set of acquisitions passing all fit gates and having finite, positive radii. The displayed accepted-fit summaries are

$$\widetilde R_{h,j}=\operatorname{median}_{i\in\mathcal A_j}(R_{h,i}), \qquad \widetilde d_{h,j}=2\widetilde R_{h,j}.$$

The median gives a robust center of the accepted acquisition estimates, but it does not quantify uncertainty, correct systematic bias, or turn repeated acquisitions from one preparation into independent sample replicates. The PDI plot likewise summarizes accepted acquisitions only; the raw curves and audit table retain information hidden by either median. Compare accepted counts and within-block spread before interpreting these summaries.

The final code cell exports one row per acquisition—including rejected fits, rejection reasons, inclusion flags, source path, wavelength, fitting threshold, optical assumptions, and viscosity—to a CSV named after the input file with `_plot_data.csv` appended. By default it writes beside the `.exp` file and replaces an existing file at that path. This makes the numeric plot data traceable, but it is a separate export from the PNG plots and summary produced by `analyze_dls.py`.

In [ ]:
def export_dls_plot_data(output_path=None):
    """Export all acquisition-level PDI and size estimates, including rejected fits."""
    import numpy as np
    from dls_walkthrough import (
        WATER_VISCOSITY_TEMPERATURES_C,
        WATER_VISCOSITIES_PA_S,
    )

    export_data = fit_table.copy()
    pdi = pd.to_numeric(export_data["pdi"], errors="coerce")
    radius = pd.to_numeric(export_data["hydrodynamic_radius_nm"], errors="coerce")
    accepted = export_data["fit_accepted"].astype(bool)
    finite_pdi = np.isfinite(pdi)
    valid_radius = np.isfinite(radius) & radius.gt(0)

    export_data["included_in_analysis"] = accepted
    export_data["excluded_from_summary"] = ~accepted
    export_data["pdi_outlier"] = pdi.notna() & ~pdi.between(0, 1)
    export_data["included_in_pdi_plot"] = accepted & finite_pdi
    export_data["included_in_radius_plot"] = valid_radius
    export_data["size_estimate_available"] = valid_radius
    export_data["source_file"] = str(DATA_PATH)
    export_data["wavelength_nm"] = experiment.wavelength_nm
    export_data["minimum_fit_r_squared"] = MIN_R_SQUARED
    export_data["scattering_angle_deg"] = SCATTERING_ANGLE_DEG
    export_data["refractive_index"] = REFRACTIVE_INDEX
    export_data["viscosity_pa_s"] = np.interp(
        export_data["temperature_c"],
        WATER_VISCOSITY_TEMPERATURES_C,
        WATER_VISCOSITIES_PA_S,
    )
    export_data["viscosity_model"] = "Water viscosity interpolated by acquisition temperature"

    if output_path is None:
        output_path = DATA_PATH.with_name(f"{DATA_PATH.stem}_plot_data.csv")
    output_path = Path(output_path)
    export_data.to_csv(output_path, index=False)
    print(f"Exported {len(export_data)} acquisition rows to {output_path}")
    return export_data


plot_data_export = export_dls_plot_data()